# Garbage Classification — K-Fold Cross Validation

**Akış:**
1. `dataset/raw/` → tüm path + label listesi
2. Stratified holdout split: **%85 trainval / %15 test** — `seed=42`, 1 kez, sabit
3. `StratifiedKFold(n_splits=5)` yalnızca **trainval** üzerinde:
   - PathDataset (path-based, ImageFolder değil)
   - WeightedRandomSampler + trash augmentation
   - ResNet18 (her fold'da fresh pretrained)
   - AMP + EarlyStopping + CosineAnnealingLR
   - W&B run: `group='kfold-v1'`, `name='fold-k'`
4. Sonuç: **mean +/- std** val accuracy (5 fold)
5. Final model: tüm trainval, k-fold ortalama epoch, holdout test raporu
6. W&B final summary tablosu

> **Test seti k-fold'a kesinlikle dahil edilmez.** Sadece final modelin son degerlendirmesinde kullanilir.

In [ ]:
import os
import random
import numpy as np
import torch
from pathlib import Path

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

In [ ]:
SOURCE_DIR  = '../data/raw'
class_names = sorted(os.listdir(SOURCE_DIR))
class_to_idx = {c: i for i, c in enumerate(class_names)}
TRASH_IDX    = class_to_idx['trash']

valid_ext = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
all_paths, all_labels = [], []

for cls in class_names:
    cls_dir = os.path.join(SOURCE_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if Path(fname).suffix.lower() in valid_ext:
            all_paths.append(os.path.join(cls_dir, fname))
            all_labels.append(class_to_idx[cls])

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels, dtype=np.int64)

print(f'Toplam: {len(all_paths)} goruntu  |  {len(class_names)} sinif')
print(f'{"Sinif":<14} {"Adet":>5}  {"%":>5}')
print('-' * 28)
for cls in class_names:
    n = (all_labels == class_to_idx[cls]).sum()
    print(f'{cls:<14} {n:>5}  {100 * n / len(all_labels):>4.1f}%')

In [ ]:
from sklearn.model_selection import train_test_split

tv_paths, test_paths, tv_labels, test_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.15,
    stratify=all_labels,
    random_state=SEED,
)

print(f'TrainVal : {len(tv_paths):>4} goruntu  (k-fold buradan calisir)')
print(f'Holdout  : {len(test_paths):>4} goruntu  (sadece final degerlendirmede)')
print()
print('Holdout sinif dagilimi:')
for cls in class_names:
    idx = class_to_idx[cls]
    n   = (test_labels == idx).sum()
    print(f'  {cls:<12} {n:>3}  ({100 * n / len(test_labels):.1f}%)')

In [ ]:
from torchvision import transforms

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

trash_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Transforms tanimlandi.')

In [ ]:
from torch.utils.data import Dataset
from PIL import Image


class PathDataset(Dataset):
    """
    ImageFolder yerine (path, label) listesinden yukler.
    train=True → trash sinifina ekstra augmentation uygulanir.
    """
    def __init__(self, paths, labels, train=True):
        self.paths   = paths
        self.labels  = labels.astype(np.int64)
        self.train   = train
        self.targets = self.labels  # WeightedRandomSampler uyumlulugu

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img   = Image.open(self.paths[idx]).convert('RGB')
        label = int(self.labels[idx])
        if self.train and label == TRASH_IDX:
            return trash_transform(img), label
        elif self.train:
            return base_transform(img), label
        else:
            return val_test_transform(img), label


class EarlyStopping:
    """
    Val loss 'patience' epoch boyunca iyilesmezse durur.
    En iyi agirlik degerlerini CPU'da saklar; restore() ile geri yukler.
    """
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best       = None
        self.counter    = 0
        self.best_epoch = 0
        self.best_state = None

    def step(self, val_loss, model, epoch):
        if self.best is None or val_loss < self.best - self.min_delta:
            self.best       = val_loss
            self.counter    = 0
            self.best_epoch = epoch
            self.best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)


print('PathDataset + EarlyStopping tanimlandi.')

In [ ]:
try:
    import wandb
except ImportError:
    import subprocess
    subprocess.run(['pip', 'install', 'wandb', '-q'], check=True)
    import wandb

PROJECT      = 'garbage-classification'
KFOLD_GROUP  = 'kfold-v1'

wandb.login()
print(f'W&B {wandb.__version__}  |  project={PROJECT!r}  group={KFOLD_GROUP!r}')

In [ ]:
N_SPLITS   = 5
MAX_EPOCHS = 30
PATIENCE   = 5
LR         = 1e-4
BATCH_SIZE = 32

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

print(f'Device: {device}  |  AMP: {use_amp}')
print(f'N_SPLITS={N_SPLITS}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}  LR={LR}  BS={BATCH_SIZE}')

## K-Fold Egitim Dongusu

- Her fold: fresh ResNet18 + WeightedRandomSampler + AMP + EarlyStopping
- W&B: her fold ayri bir run (`group='kfold-v1'`)
- Fold bittikten sonra best checkpoint restore edilir, val accuracy o noktada olcutur

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import models
import torch.nn as nn
import torch.optim as optim

skf          = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
fold_results = []

for fold_k, (train_idx, val_idx) in enumerate(skf.split(tv_paths, tv_labels)):
    print(f'\n{"=" * 62}')
    print(f'  FOLD {fold_k + 1}/{N_SPLITS}  '
          f'(train={len(train_idx)}, val={len(val_idx)})')
    print(f'{"=" * 62}')

    # ── Dataset & Sampler ──────────────────────────────────
    fold_train_ds = PathDataset(tv_paths[train_idx], tv_labels[train_idx], train=True)
    fold_val_ds   = PathDataset(tv_paths[val_idx],   tv_labels[val_idx],   train=False)

    fold_counts = np.bincount(tv_labels[train_idx],
                              minlength=len(class_names)).astype(float)
    sample_w    = torch.DoubleTensor(
        [1.0 / max(fold_counts[l], 1) for l in tv_labels[train_idx]])
    sampler     = WeightedRandomSampler(
        sample_w, num_samples=len(sample_w), replacement=True)

    train_loader = DataLoader(fold_train_ds, batch_size=BATCH_SIZE,
                              sampler=sampler, num_workers=0)
    val_loader   = DataLoader(fold_val_ds,   batch_size=BATCH_SIZE,
                              shuffle=False,  num_workers=0)

    # ── Model (her fold fresh) ─────────────────────────────
    model    = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    model    = model.to(device)

    cls_w     = torch.FloatTensor(
        1.0 / np.maximum(fold_counts, 1)).to(device)
    cls_w     = cls_w / cls_w.min()
    criterion = nn.CrossEntropyLoss(weight=cls_w)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
                    optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)
    es        = EarlyStopping(patience=PATIENCE)
    scaler    = torch.cuda.amp.GradScaler(enabled=use_amp)

    # ── W&B run ────────────────────────────────────────────
    wandb.finish()
    wandb.init(
        project=PROJECT,
        group=KFOLD_GROUP,
        name=f'fold-{fold_k + 1}',
        config={
            'fold':       fold_k + 1,
            'n_splits':   N_SPLITS,
            'model':      'resnet18',
            'lr':         LR,
            'batch_size': BATCH_SIZE,
            'patience':   PATIENCE,
            'max_epochs': MAX_EPOCHS,
            'amp':        use_amp,
            'scheduler':  'CosineAnnealingLR',
            'train_size': len(train_idx),
            'val_size':   len(val_idx),
        },
        reinit=True,
    )

    # ── Epoch dongusu ──────────────────────────────────────
    for epoch in range(MAX_EPOCHS):
        model.train()
        train_loss = 0.0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        model.eval()
        val_loss = correct = total = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                with torch.autocast(device_type=device.type, enabled=use_amp):
                    out       = model(imgs)
                    val_loss += criterion(out, lbls).item()
                correct += (out.argmax(1) == lbls).sum().item()
                total   += lbls.size(0)

        scheduler.step()
        avg_tr  = train_loss / len(train_loader)
        avg_val = val_loss   / len(val_loader)
        val_acc = correct / total
        lr_now  = scheduler.get_last_lr()[0]

        print(f'  Ep {epoch + 1:2d} | '
              f'Train: {avg_tr:.4f} | Val: {avg_val:.4f} | '
              f'Acc: {val_acc:.4f} | LR: {lr_now:.2e}')

        wandb.log({
            'epoch':      epoch + 1,
            'train_loss': avg_tr,
            'val_loss':   avg_val,
            'val_acc':    val_acc,
            'lr':         lr_now,
        })

        if es.step(avg_val, model, epoch + 1):
            print(f'  Early stopping @ epoch {epoch + 1} '
                  f'(best val loss @ ep {es.best_epoch}: {es.best:.4f})')
            break

    # ── Best checkpoint → val accuracy ─────────────────────
    es.restore(model)
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            correct += (model(imgs).argmax(1).cpu() == lbls).sum().item()
            total   += lbls.size(0)
    best_val_acc = correct / total

    fold_results.append({
        'fold':          fold_k + 1,
        'best_val_acc':  best_val_acc,
        'best_epoch':    es.best_epoch,
    })

    wandb.log({
        'fold_best_val_acc': best_val_acc,
        'fold_best_epoch':   es.best_epoch,
    })
    wandb.finish()
    print(f'  --> Fold {fold_k + 1}  best val acc: {best_val_acc:.4f} '
          f'(checkpoint ep {es.best_epoch})')

## K-Fold Sonuclari

In [ ]:
import pandas as pd

df_folds   = pd.DataFrame(fold_results)
mean_acc   = df_folds['best_val_acc'].mean()
std_acc    = df_folds['best_val_acc'].std()
mean_epoch = df_folds['best_epoch'].mean()

FINAL_EPOCHS = max(5, int(round(mean_epoch)))

print('=' * 54)
print('  K-FOLD SONUCLARI')
print('=' * 54)
print(df_folds.to_string(index=False))
print('-' * 54)
print(f'  Val Acc:          {mean_acc:.4f} +/- {std_acc:.4f}')
print(f'  Ort. best epoch:  {mean_epoch:.1f}  -->  Final epochs: {FINAL_EPOCHS}')
print('=' * 54)

# W&B: kfold summary run
wandb.finish()
wandb.init(
    project=PROJECT,
    group=KFOLD_GROUP,
    name='kfold-summary',
    reinit=True,
)
wandb.log({
    'mean_val_acc':    mean_acc,
    'std_val_acc':     std_acc,
    'mean_best_epoch': mean_epoch,
    'fold_table':      wandb.Table(dataframe=df_folds),
})
wandb.finish()
print('W&B summary loglandi.')

## Final Model Egitimi

K-fold sonucunda belirlenen ortalama epoch sayisiyla **tum trainval** seti uzerinde tek model egitilir.  
Bu model sunum icin asil modeldir — holdout test seti yalnizca burada kullanilir.

In [ ]:
print(f'Final model: {len(tv_paths)} ornek, {FINAL_EPOCHS} epoch')
print('(Test seti egitimde kullanilmiyor)\n')

# Dataset + Sampler
final_ds     = PathDataset(tv_paths, tv_labels, train=True)
all_counts   = np.bincount(tv_labels, minlength=len(class_names)).astype(float)
sample_w_f   = torch.DoubleTensor(
    [1.0 / max(all_counts[l], 1) for l in tv_labels])
sampler_f    = WeightedRandomSampler(
    sample_w_f, num_samples=len(sample_w_f), replacement=True)
final_loader = DataLoader(final_ds, batch_size=BATCH_SIZE,
                          sampler=sampler_f, num_workers=0)

# Holdout test loader
test_ds      = PathDataset(test_paths, test_labels, train=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

# Model
final_model    = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
final_model.fc = nn.Linear(final_model.fc.in_features, len(class_names))
final_model    = final_model.to(device)

cls_w_f      = torch.FloatTensor(1.0 / np.maximum(all_counts, 1)).to(device)
cls_w_f      = cls_w_f / cls_w_f.min()
criterion_f  = nn.CrossEntropyLoss(weight=cls_w_f)
optimizer_f  = optim.Adam(final_model.parameters(), lr=LR)
scheduler_f  = optim.lr_scheduler.CosineAnnealingLR(
                   optimizer_f, T_max=FINAL_EPOCHS, eta_min=1e-6)
scaler_f     = torch.cuda.amp.GradScaler(enabled=use_amp)

# W&B run
wandb.finish()
wandb.init(
    project=PROJECT,
    group=KFOLD_GROUP,
    name='final-model',
    config={
        'model':          'resnet18',
        'train_size':     len(tv_paths),
        'test_size':      len(test_paths),
        'final_epochs':   FINAL_EPOCHS,
        'lr':             LR,
        'batch_size':     BATCH_SIZE,
        'kfold_mean_acc': round(mean_acc, 4),
        'kfold_std_acc':  round(std_acc, 4),
        'amp':            use_amp,
    },
    reinit=True,
)

for epoch in range(FINAL_EPOCHS):
    final_model.train()
    train_loss = 0.0
    for imgs, lbls in final_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer_f.zero_grad()
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = criterion_f(final_model(imgs), lbls)
        scaler_f.scale(loss).backward()
        scaler_f.step(optimizer_f)
        scaler_f.update()
        train_loss += loss.item()
    scheduler_f.step()
    avg_f  = train_loss / len(final_loader)
    lr_now = scheduler_f.get_last_lr()[0]
    print(f'Epoch {epoch + 1:2d}/{FINAL_EPOCHS} | Loss: {avg_f:.4f} | LR: {lr_now:.2e}')
    wandb.log({'epoch': epoch + 1, 'train_loss': avg_f, 'lr': lr_now})

print('\nFinal model egitimi tamamlandi.')

## Holdout Test Degerlendirmesi

Test seti bu notebook'ta ilk ve tek kez burada kullaniliyor.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

final_model.eval()
all_preds_f, all_labels_f = [], []

with torch.no_grad():
    for imgs, lbls in test_loader:
        preds = final_model(imgs.to(device)).argmax(1).cpu().numpy()
        all_preds_f.extend(preds)
        all_labels_f.extend(lbls.numpy())

all_preds_f  = np.array(all_preds_f)
all_labels_f = np.array(all_labels_f)
test_acc     = float((all_preds_f == all_labels_f).mean())

print(f'Holdout Test Accuracy: {test_acc:.4f}\n')
print(classification_report(all_labels_f, all_preds_f, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(all_labels_f, all_preds_f)
fig_cm, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_title('Confusion Matrix — Holdout Test (Final Model)')
ax.set_xlabel('Tahmin')
ax.set_ylabel('Gercek')
plt.tight_layout()
plt.show()

# W&B: final summary
wandb.log({
    'holdout_test_accuracy': test_acc,
    'confusion_matrix': wandb.plot.confusion_matrix(
        y_true=all_labels_f.tolist(),
        preds=all_preds_f.tolist(),
        class_names=class_names,
    ),
    'confusion_matrix_fig': wandb.Image(fig_cm),
    'final_summary': wandb.Table(
        columns=['Metric', 'Value'],
        data=[
            ['K-Fold Val Acc (mean)', f'{mean_acc:.4f}'],
            ['K-Fold Val Acc (std)',  f'{std_acc:.4f}'],
            ['Holdout Test Acc',      f'{test_acc:.4f}'],
            ['TrainVal Size',         str(len(tv_paths))],
            ['Holdout Test Size',     str(len(test_paths))],
            ['K-Fold Avg Best Epoch', f'{mean_epoch:.1f}'],
            ['Final Epochs Used',     str(FINAL_EPOCHS)],
        ],
    ),
})
wandb.finish()

print(f'\n{"=" * 50}')
print(f'  K-Fold Val:    {mean_acc:.4f} +/- {std_acc:.4f}')
print(f'  Holdout Test:  {test_acc:.4f}')
print(f'{"=" * 50}')